In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataCleaning").getOrCreate()

df = spark.read.csv(
    "customers_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

In [ ]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, ArrayType

def flatten_df(df):
    
    def _flatten(schema, prefix=""):
        fields = []
        for field in schema.fields:
            name = f"{prefix}{field.name}"
            dtype = field.dataType
            if isinstance(dtype, StructType):
                # recurse into struct
                fields.extend(_flatten(dtype, prefix=name + "."))
            elif isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType):
                # explode array of structs
                fields.append((name, "explode"))
            else:
                fields.append((name, "column"))
        return fields

    flat_fields = _flatten(df.schema)

    # Explode arrays first
    for f, t in flat_fields:
        if t == "explode":
            df = df.withColumn(f, explode(col(f)))

    # Build select expressions
    select_exprs = []
    for f, t in flat_fields:
        if t == "column":
            select_exprs.append(col(f).alias(f.replace(".", "_")))
        elif t == "explode":
            # flatten struct fields inside exploded array
            struct_fields = df.select(col(f + ".*")).schema.fields
            for sf in struct_fields:
                select_exprs.append(col(f + "." + sf.name).alias(f.replace(".", "_") + "_" + sf.name))

    return df.select(*select_exprs)


In [ ]:
nested_df = spark.read.json("nested_orders.json")

flat_df = flatten_df(nested_df)
flat_df.printSchema()
flat_df.show(truncate=False)


In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkOptimization") \
    .getOrCreate()

skewed_df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)

In [ ]:
# print(orders.rdd.getNumPartitions())

In [ ]:
orders_repart = orders.repartition(8)

print(orders_repart.rdd.getNumPartitions())

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id

spark = SparkSession.builder.getOrCreate()

df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)
df.show()

+------+-----------+------+------------+-------+
|txn_id|customer_id|amount|payment_mode| status|
+------+-----------+------+------------+-------+
|     1|        101|  2944|         UPI|SUCCESS|
|     2|        560|  3924|      WALLET|SUCCESS|
|     3|        101|  4782|      WALLET|SUCCESS|
|     4|        101|  4863|        CARD| FAILED|
|     5|        101|  1694|        CARD|SUCCESS|
|     6|        101|  2859|         UPI| FAILED|
|     7|        101|  2372|  NETBANKING| FAILED|
|     8|        101|   479|         UPI| FAILED|
|     9|        101|  3554|      WALLET|SUCCESS|
|    10|        101|   891|         UPI| FAILED|
|    11|        101|  3417|        CARD|SUCCESS|
|    12|       3998|   194|  NETBANKING| FAILED|
|    13|        101|  2421|         UPI|SUCCESS|
|    14|       7837|  2516|         UPI| FAILED|
|    15|        101|   804|  NETBANKING| FAILED|
|    16|        101|  2364|      WALLET|SUCCESS|
|    17|        101|  4877|         UPI|SUCCESS|
|    18|        101|

In [2]:
print(df.rdd.getNumPartitions())

3


In [3]:
partition_distribution = df.withColumn(
    "partition_id",
    spark_partition_id()
)

partition_distribution.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show(truncate=False)

+------------+------+
|partition_id|count |
+------------+------+
|0           |137848|
|1           |134249|
|2           |27903 |
+------------+------+



In [4]:
repart_df = df.repartition(8)

In [5]:
repart_df.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy("partition_id") \
 .count() \
 .orderBy("partition_id") \
 .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|37500|
|           1|37500|
|           2|37501|
|           3|37500|
|           4|37500|
|           5|37500|
|           6|37500|
|           7|37499|
+------------+-----+



In [6]:
#coalesce()

df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)

df = df.repartition(10)

print(df.rdd.getNumPartitions())
#10 files will be created when we write it

10


In [7]:
small_df = df.coalesce(3)

print(small_df.rdd.getNumPartitions())

3


In [8]:
small_df.write.mode("overwrite").csv("few_files")

In [9]:
#IMBALNCE
from pyspark.sql.functions import spark_partition_id

small_df.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy("partition_id") \
 .count() \
 .show()

+------------+------+
|partition_id| count|
+------------+------+
|           0|120001|
|           1| 90000|
|           2| 89999|
+------------+------+



In [ ]:
#BROADCASTING

#BEFORE
joined = orders.join(products, "product_id")

joined.explain(True)

In [ ]:
from pyspark.sql.functions import broadcast

broadcast_joined = orders.join(
    broadcast(products),
    "product_id"
)

broadcast_joined.explain(True)

In [34]:
#COMPARING NORMAL and BROADCAST
#ORDERS DATASET
from pyspark.sql import SparkSession
import random
import pandas as pd

spark = SparkSession.builder \
    .appName("BroadcastJoinDemo") \
    .getOrCreate()

orders = []

for i in range(1, 1000001):

    orders.append([
        i,
        random.randint(1, 500),      # product_id
        random.randint(1, 5),        # quantity
        random.randint(500, 10000)   # amount
    ])

orders_pdf = pd.DataFrame(
    orders,
    columns=[
        "order_id",
        "product_id",
        "quantity",
        "amount"
    ]
)

skewed_df = spark.read.csv(
    "skewed_transactions.csv",
    header=True,
    inferSchema=True
)

#customers dataset
customers_pdf = []
for i in range(1, 1001):
    customers_pdf.append([
        i,
        f"Customer_{i}",
        f"customer_{i}@example.com"
    ])

# use 'schema' instead of unsupported 'columns' keyword
customers_df = spark.createDataFrame(customers_pdf, schema=["customer_id", "customer_name", "email"])
customers_df.createOrReplaceTempView("customers")
orders_df = spark.createDataFrame(orders_pdf)

In [35]:
#PRODUCTS DATASET
products = []

categories = [
    "Electronics",
    "Fashion",
    "Books",
    "Home",
    "Sports"
]

for i in range(1, 501):

    products.append([
        i,
        f"Product_{i}",
        random.choice(categories),
        random.randint(100, 5000)
    ])

products_pdf = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "product_name",
        "category",
        "price"
    ]
)

products_df = spark.createDataFrame(products_pdf)

In [36]:
normal_join = orders_df.join(
    products_df,
    "product_id"
)
from pyspark.sql.functions import broadcast

broadcast_join = orders_df.join(
    broadcast(products_df),
    "product_id"
)

In [37]:
import time

# NORMAL JOIN

start = time.time()
normal_join.count()
end = time.time()

print("Normal Join Time:", end - start)

# BROADCAST JOIN
start = time.time()
broadcast_join.count()
end = time.time()

print("Broadcast Join Time:", end - start)

#28.7 seconds -------------normal
#5.3 seconds ------------broadcast 

Normal Join Time: 14.593893051147461
Broadcast Join Time: 14.075180530548096


In [38]:
#SALTING
skewed_df.groupBy("customer_id").count().show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|        833|    6|
|       9900|    9|
|       4101|    7|
|        148|    6|
|       9427|   10|
|       7754|    8|
|       7833|    2|
|       6357|    7|
|       3997|    8|
|       1238|    8|
|       4818|    7|
|       5803|    4|
|       9465|    9|
|       5156|    3|
|       5300|    6|
|       7982|    6|
|       7554|    6|
|       6620|    5|
|       6658|    6|
|       3175|    9|
+-----------+-----+
only showing top 20 rows


In [40]:
joined = skewed_df.join(customers_df, "customer_id")
joined.show(100)

+-----------+------+------+------------+-------+-------------+--------------------+
|customer_id|txn_id|amount|payment_mode| status|customer_name|               email|
+-----------+------+------+------------+-------+-------------+--------------------+
|          1|296484|  4627|        CARD|SUCCESS|   Customer_1|customer_1@exampl...|
|          1|215476|  1362|      WALLET| FAILED|   Customer_1|customer_1@exampl...|
|          1|214236|  3930|      WALLET|SUCCESS|   Customer_1|customer_1@exampl...|
|          1|119986|   392|      WALLET| FAILED|   Customer_1|customer_1@exampl...|
|          1|  9533|  3895|        CARD|SUCCESS|   Customer_1|customer_1@exampl...|
|          2|279736|  3896|         UPI| FAILED|   Customer_2|customer_2@exampl...|
|          2|168165|  3592|  NETBANKING| FAILED|   Customer_2|customer_2@exampl...|
|          2| 91819|  1998|         UPI|SUCCESS|   Customer_2|customer_2@exampl...|
|          2| 48860|   532|      WALLET| FAILED|   Customer_2|customer_2@exa

In [41]:
before_partition_data = (
    skewed_df.rdd
    .mapPartitions(lambda x: [sum(1 for _ in x)])
    .collect()
)

print(before_partition_data)
[198656,45,49,65]

[137848, 134249, 27903]


[198656, 45, 49, 65]

In [42]:
#ADDING SALT COLUMN
from pyspark.sql.functions import floor, rand, concat_ws

salted_skewed = skewed_df.withColumn(
    "salt",
    floor(rand() * 5)
)

In [43]:
#DUPLICATE SMALL TABLE
from pyspark.sql.functions import explode, array, lit

expanded_customers = customers_df.withColumn(
    "salt",
    explode(array(
        lit(0),
        lit(1),
        lit(2),
        lit(3),
        lit(4)
    ))
)

In [44]:
salted_join = salted_skewed.join(
    expanded_customers,
    ["customer_id", "salt"]
)

In [45]:
salted_join.show(10)

+-----------+----+------+------+------------+-------+-------------+--------------------+
|customer_id|salt|txn_id|amount|payment_mode| status|customer_name|               email|
+-----------+----+------+------+------------+-------+-------------+--------------------+
|        309|   4|  2997|  3283|         UPI| FAILED| Customer_309|customer_309@exam...|
|        713|   0|  4966|  2678|         UPI| FAILED| Customer_713|customer_713@exam...|
|        255|   3|  5235|  3735|         UPI|SUCCESS| Customer_255|customer_255@exam...|
|        908|   2| 17786|  4876|      WALLET| FAILED| Customer_908|customer_908@exam...|
|        380|   4| 34701|  3397|  NETBANKING|SUCCESS| Customer_380|customer_380@exam...|
|        713|   0| 44762|  3180|         UPI| FAILED| Customer_713|customer_713@exam...|
|        901|   0| 54732|  1150|         UPI|SUCCESS| Customer_901|customer_901@exam...|
|        615|   1| 66319|  1080|        CARD| FAILED| Customer_615|customer_615@exam...|
|          3|   0| 71

In [46]:
#repartitioning based on salt
after_partition_data = (
    salted_join.repartition("customer_id", "salt")
    .rdd
    .mapPartitions(lambda x: [sum(1 for _ in x)])
    .collect()
)

print(after_partition_data)
[85,77,65]

[96694, 50393, 49674, 49408]


[85, 77, 65]

In [47]:
#CACHE

from pyspark.sql.functions import sum

sales_df = orders_df.groupBy("product_id") \
    .agg(
        sum("amount").alias("total_sales")
    )

In [48]:
#WITHOUT CACHE
import time
start = time.time()
sales_df.show()
sales_df.count()
sales_df.orderBy("total_sales", ascending=False).show()
end = time.time()

print("Without Cache Time:", end - start)

+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|        29|   10580873|
|       474|   10343208|
|        26|   10576071|
|       418|   10185406|
|        65|   10503370|
|       191|   10061119|
|       293|   10627621|
|       270|   10329102|
|       222|   10394465|
|       442|   10589148|
|       278|   10495789|
|       243|   10007410|
|       367|   10393883|
|        54|   10092940|
|       296|   10455522|
|        19|   10335112|
|       277|   10156647|
|       348|   10437620|
|       287|    9861144|
|       415|   10752033|
+----------+-----------+
only showing top 20 rows
+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|       485|   11212843|
|       118|   11177950|
|       377|   11158694|
|       148|   11091516|
|       422|   11059163|
|       395|   11041677|
|       369|   11022153|
|       328|   11019819|
|        57|   11005311|
|        97|   10997449|
|       242|   10995784|
|       461|   10951990|


In [49]:
#apply cache
sales_df.cache()
sales_df.count()

500

In [50]:
#RE-EXECUTING IT
start = time.time()
sales_df.show()
sales_df.count()
sales_df.orderBy("total_sales", ascending=False).show()
end = time.time()
print("With Cache Time:", end - start)

+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|        29|   10580873|
|       474|   10343208|
|        26|   10576071|
|       418|   10185406|
|        65|   10503370|
|       191|   10061119|
|       293|   10627621|
|       270|   10329102|
|       222|   10394465|
|       442|   10589148|
|       278|   10495789|
|       243|   10007410|
|       367|   10393883|
|        54|   10092940|
|       296|   10455522|
|        19|   10335112|
|       277|   10156647|
|       348|   10437620|
|       287|    9861144|
|       415|   10752033|
+----------+-----------+
only showing top 20 rows
+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|       485|   11212843|
|       118|   11177950|
|       377|   11158694|
|       148|   11091516|
|       422|   11059163|
|       395|   11041677|
|       369|   11022153|
|       328|   11019819|
|        57|   11005311|
|        97|   10997449|
|       242|   10995784|
|       461|   10951990|


In [51]:
print(sales_df.is_cached)

True


In [ ]:
from pyspark import StorageLevel

sales_df.persist(StorageLevel.MEMORY_AND_DISK)

sales_df.count()

500

In [54]:
#PERSIST   - remaining partitions stored on disk
from pyspark import StorageLevel
sales_df.persist(StorageLevel.MEMORY_ONLY)

sales_df.count()

500

In [55]:
sales_df.persist(StorageLevel.MEMORY_AND_DISK)

sales_df.count()

500

In [56]:
sales_df.persist(StorageLevel.DISK_ONLY)

sales_df.count()

500